# Lab type: review
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: Clustering in Production
# Task: Evaluate the production clustering pipeline. Answer the judgment questions.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib
import json

## Step 1: Generate Sample Customer Data

In [ ]:
np.random.seed(42)

# Simulate three months of customer data
month_1 = pd.DataFrame({
    'customer_id': range(1, 101),
    'annual_spend': np.random.uniform(500, 50000, 100),
    'visits_per_year': np.random.poisson(20, 100),
    'account_age_months': np.random.uniform(0, 60, 100)
})

print(f"Month 1 data shape: {month_1.shape}")
print(month_1.head())

## Step 2: Review Training and Serialization

In [ ]:
# Review this code — is it production-ready?

# Step 1: Fit scaler and model on Month 1 data
X_month_1 = month_1[['annual_spend', 'visits_per_year', 'account_age_months']].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_month_1)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

month_1['cluster'] = labels

print(f"Month 1 cluster distribution:")
print(month_1['cluster'].value_counts().sort_index())

# Save model and scaler for later use
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(kmeans, 'kmeans_model.pkl')
print("\nModel and scaler saved.")


**Question:** When you save the scaler and model, what happens when new customers arrive next month? Should you retrain the scaler? Retrain the kmeans model? Or use the same scaler and model?

## Step 3: Review Prediction on New Data (Month 2)

In [ ]:
# Simulate new customer data arriving in Month 2
month_2 = pd.DataFrame({
    'customer_id': range(101, 121),
    'annual_spend': np.random.uniform(500, 50000, 20),
    'visits_per_year': np.random.poisson(20, 20),
    'account_age_months': np.random.uniform(0, 60, 20)
})

print(f"Month 2 data shape: {month_2.shape}")

# Load saved scaler and model
loaded_scaler = joblib.load('scaler.pkl')
loaded_kmeans = joblib.load('kmeans_model.pkl')

# Apply clustering to new data
X_month_2 = month_2[['annual_spend', 'visits_per_year', 'account_age_months']].values
X_month_2_scaled = loaded_scaler.transform(X_month_2)  # Note: transform, not fit_transform!
labels_month_2 = loaded_kmeans.predict(X_month_2_scaled)

month_2['cluster'] = labels_month_2

print(f"Month 2 cluster distribution:")
print(month_2['cluster'].value_counts().sort_index())

**Question:** The cluster assignments look different between Month 1 and Month 2. Is this expected? How would you know if the change is due to real shifts in customer behavior vs. algorithmic drift?

## Step 4: Review Monitoring and Retraining Strategy

In [ ]:
# Review this code — what should trigger a retrain?

print("Monitoring strategy:")
print("\n1. Track cluster sizes over time")
print(f"   Month 1: {dict(month_1['cluster'].value_counts().sort_index())}")
print(f"   Month 2: {dict(month_2['cluster'].value_counts().sort_index())}")
print("   Change? If one cluster shrinks while another grows, investigate.")

print("\n2. Track feature distributions")
print(f"   Month 1 spend mean: ${month_1['annual_spend'].mean():.0f}")
print(f"   Month 2 spend mean: ${month_2['annual_spend'].mean():.0f}")
print("   If distributions shift significantly, the scaler may become invalid.")

print("\n3. Retraining decision rules:")
print("   - If any cluster shrinks to < 5% of total, retrain")
print("   - If feature distributions shift > 10% from original, retrain")
print("   - If business defines new segmentation goals, retrain")
print("   - Quarterly retraining as a safety fallback")


**Question:** What's the cost of retraining too often (monthly) vs. too infrequently (yearly)? When would you choose each approach?

## Step 5: Review Cluster Interpretation and Documentation

In [ ]:
# Review this code — is the cluster interpretation documented?

print("Cluster Interpretation (Month 1):")
for cluster_id in sorted(month_1['cluster'].unique()):
    cluster_data = month_1[month_1['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id}:")
    print(f"  Size: {len(cluster_data)} customers")
    print(f"  Avg spend: ${cluster_data['annual_spend'].mean():.0f}")
    print(f"  Avg visits/year: {cluster_data['visits_per_year'].mean():.1f}")
    print(f"  Avg account age: {cluster_data['account_age_months'].mean():.1f} months")

# Save interpretation as metadata
metadata = {
    'model_version': '1.0',
    'training_date': '2026-05-07',
    'n_clusters': 3,
    'scaler_type': 'StandardScaler',
    'features': ['annual_spend', 'visits_per_year', 'account_age_months'],
    'interpretation': {
        '0': 'High-value, frequent customers',
        '1': 'Moderate-value, occasional customers',
        '2': 'Low-value, new accounts'
    }
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\nModel metadata saved for documentation.")


**Question:** Why is it important to document the cluster interpretation alongside the model? What happens if the interpretation is lost?

## Summary

Clustering in production requires:
- **Separation of concerns:** Use `fit_transform` on training data, `transform` on new data
- **Versioning:** Save scaler and model together
- **Monitoring:** Track cluster sizes and feature distributions
- **Retraining strategy:** Decide when to retrain based on data drift
- **Documentation:** Save interpretation alongside the model
- **Validation:** Compare Month 1 and Month 2 assignments; investigate large shifts

Clustering is never truly "done." Plan for ongoing monitoring and periodic retraining from the start.

**End of course:** You've covered the full clustering workflow—from understanding distance to deploying in production.